# 🏦 Bank Customer Churn Prediction

**Goal:** Predict whether a customer will leave (`churn`) the bank, using a classification model built with scikit-learn.

**Pipeline:**
1. Load & explore the data
2. Clean & preprocess (missing values, encoding, scaling)
3. Train/test split
4. Build & evaluate classification models
5. Interpret results -> business recommendations

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)

RANDOM_STATE = 42

In [ ]:
# Download and unzip the dataset directly inside Colab
!wget -q "https://github.com/devtlv/Datasets-GEN-AI-Bootcamp/raw/refs/heads/main/Week%205/Day%204%20-%20Statistics%20for%20Machine%20Learning/Inferential%20Statistics.zip" -O data.zip
!unzip -o -q data.zip -d data
!find data -type f

In [ ]:
# TODO: update this path to match the CSV filename printed by the `find` command above
csv_path = "data/Churn_Modelling.csv"  # <-- adjust if needed

df = pd.read_csv(csv_path)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## 2. Exploratory Data Analysis

In [ ]:
# TODO: identify the target column name (commonly 'Exited' or 'Churn')
target_col = "Exited"  # <-- adjust if needed

churn_rate = df[target_col].mean() * 100
print(f"Overall churn rate: {churn_rate:.2f}%")

sns.countplot(x=target_col, data=df)
plt.title("Churn Distribution (0 = Stayed, 1 = Churned)")
plt.show()

In [ ]:
# Churn vs key numeric features
numeric_cols = [c for c in ["Age", "Balance", "CreditScore", "Tenure", "NumOfProducts", "EstimatedSalary"] if c in df.columns]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.flatten(), numeric_cols):
    sns.boxplot(x=target_col, y=col, data=df, ax=ax)
    ax.set_title(f"{col} vs Churn")
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap (numeric features only)
plt.figure(figsize=(10, 8))
sns.heatmap(df.select_dtypes(include=np.number).corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

## 3. Data Preprocessing

In [ ]:
# 3.1 Drop columns that are pure identifiers (no predictive value)
id_cols = [c for c in ["RowNumber", "CustomerId", "Surname"] if c in df.columns]
df_model = df.drop(columns=id_cols)
df_model.head()

In [ ]:
# 3.2 Handle missing values
print("Missing values per column:\n", df_model.isnull().sum())

# TODO: For numeric columns, fill missing values with the median.
# For categorical columns, fill missing values with the mode.
for col in df_model.columns:
    if df_model[col].isnull().sum() > 0:
        if df_model[col].dtype in ["int64", "float64"]:
            df_model[col] = df_model[col].fillna(df_model[col].median())
        else:
            df_model[col] = df_model[col].fillna(df_model[col].mode()[0])

print("\nRemaining missing values:", df_model.isnull().sum().sum())

In [ ]:
# 3.3 Encode categorical features
categorical_cols = df_model.select_dtypes(include="object").columns.tolist()
print("Categorical columns to encode:", categorical_cols)

# TODO: One-hot encode categorical columns (drop_first avoids the dummy-variable trap)
df_encoded = pd.get_dummies(df_model, columns=categorical_cols, drop_first=True)
df_encoded.head()

In [ ]:
# 3.4 Split features / target
X = df_encoded.drop(columns=[target_col])
y = df_encoded[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

In [ ]:
# 3.5 Feature scaling (important for Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 4. Model Building

In [ ]:
# 4.1 Logistic Regression (baseline, interpretable model)
log_reg = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
log_reg.fit(X_train_scaled, y_train)
y_pred_lr = log_reg.predict(X_test_scaled)
y_proba_lr = log_reg.predict_proba(X_test_scaled)[:, 1]

In [ ]:
# 4.2 Random Forest (stronger, non-linear model)
rf = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=RANDOM_STATE)
rf.fit(X_train, y_train)  # tree models don't need scaling
y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

## 5. Model Evaluation

In [ ]:
def evaluate(name, y_true, y_pred, y_proba):
    print(f"--- {name} ---")
    print("Accuracy :", round(accuracy_score(y_true, y_pred), 3))
    print("Precision:", round(precision_score(y_true, y_pred), 3))
    print("Recall   :", round(recall_score(y_true, y_pred), 3))
    print("F1-score :", round(f1_score(y_true, y_pred), 3))
    print("ROC-AUC  :", round(roc_auc_score(y_true, y_proba), 3))
    print()
    print(classification_report(y_true, y_pred))

evaluate("Logistic Regression", y_test, y_pred_lr, y_proba_lr)
evaluate("Random Forest", y_test, y_pred_rf, y_proba_rf)

In [ ]:
# Confusion matrices side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (name, y_pred) in zip(axes, [("Logistic Regression", y_pred_lr), ("Random Forest", y_pred_rf)]):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["Stayed", "Churned"], yticklabels=["Stayed", "Churned"])
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.show()

In [ ]:
# ROC curves
plt.figure(figsize=(7, 6))
for name, y_proba in [("Logistic Regression", y_proba_lr), ("Random Forest", y_proba_rf)]:
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.2f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.show()

## 6. Feature Importance & Business Insights

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(8, 6))
sns.barplot(x=importances.values[:10], y=importances.index[:10])
plt.title("Top 10 Feature Importances (Random Forest)")
plt.xlabel("Importance")
plt.show()

importances.head(10)

In [ ]:
# Identify the highest-risk customers in the test set
results = X_test.copy()
results["actual_churn"] = y_test.values
results["churn_probability"] = y_proba_rf

high_risk = results.sort_values("churn_probability", ascending=False).head(20)
high_risk[["churn_probability", "actual_churn"]]

### 📊 Business Takeaways (fill in after running the notebook)

- **TODO:** Which features matter most for predicting churn (e.g., Age, NumOfProducts, IsActiveMember)?
- **TODO:** What customer segments (age range, balance range, product count) show the highest churn risk?
- **TODO:** Based on `churn_probability`, propose 2-3 concrete retention actions the bank could take (e.g., targeted offers for customers with only 1 product, proactive outreach for inactive members).
- **TODO:** Compare precision vs recall trade-offs: would the bank rather over-flag at-risk customers (high recall) or only flag the most certain cases (high precision)? Justify your choice.